In [ ]:
# Gerekli kurulumlar
!pip install xgboost shap tensorflow -q

# Kütüphaneler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

# Dosya yükleme (Colab uyumlu)
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# Veri okuma
df = pd.read_csv(file_name)

# Zaman düzenleme
df["Time"] = pd.to_datetime(df["Time"])
df = df.sort_values("Time").reset_index(drop=True)

# Gereksiz sütunları sil
drop_cols = [
    "DailyAP", "DailyIrrediation", "Year", "Month", "month",
    "Minute", "Season", "MonthIndex", "date",
    "snowfall_height", "lightning_potential"
]
df = df.drop(columns=drop_cols, errors="ignore")

# Lag feature'lar
df["lag_1"] = df["AP"].shift(1)
df["lag_4"] = df["AP"].shift(4)
df["lag_8"] = df["AP"].shift(8)
df["lag_96"] = df["AP"].shift(96)

df = df.dropna()

# Özellikler ve hedef değişken
y = df["AP"]
X = df.drop(columns=["AP", "Time"])

# Train-test split
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# Random Forest
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("RF MAE:", mae_rf)
print("RF RMSE:", rmse_rf)

plt.figure(figsize=(12, 5))
plt.plot(y_test.values[:500], label="Actual")
plt.plot(y_pred_rf[:500], label="RF")
plt.legend()
plt.title("Random Forest Prediction")
plt.show()

# XGBoost
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

print("XGB MAE:", mae_xgb)
print("XGB RMSE:", rmse_xgb)

# RF ve XGB karşılaştırması
models = ["RF", "XGB"]
mae_vals = [mae_rf, mae_xgb]
rmse_vals = [rmse_rf, rmse_xgb]

x = np.arange(len(models))
w = 0.3

plt.figure(figsize=(8, 5))
plt.bar(x - w/2, mae_vals, w, label="MAE")
plt.bar(x + w/2, rmse_vals, w, label="RMSE")
plt.xticks(x, models)
plt.legend()
plt.title("Model Comparison")
plt.show()

# LSTM
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_s = scaler_X.fit_transform(X_train)
X_test_s = scaler_X.transform(X_test)

y_train_s = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_s = scaler_y.transform(y_test.values.reshape(-1, 1))

def seq(X, y, steps=24):
    Xs, ys = [], []
    for i in range(steps, len(X)):
        Xs.append(X[i-steps:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_tr_seq, y_tr_seq = seq(X_train_s, y_train_s)
X_te_seq, y_te_seq = seq(X_test_s, y_test_s)

model = Sequential([
    Input(shape=(X_tr_seq.shape[1], X_tr_seq.shape[2])),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse")

model.fit(
    X_tr_seq,
    y_tr_seq,
    epochs=20,
    batch_size=64,
    validation_split=0.1
)

y_pred_lstm_s = model.predict(X_te_seq)

y_pred_lstm = scaler_y.inverse_transform(y_pred_lstm_s)
y_test_real = scaler_y.inverse_transform(y_te_seq)

mae_lstm = mean_absolute_error(y_test_real, y_pred_lstm)
rmse_lstm = np.sqrt(mean_squared_error(y_test_real, y_pred_lstm))

print("LSTM MAE:", mae_lstm)
print("LSTM RMSE:", rmse_lstm)

# Tüm modellerin karşılaştırması
models = ["RF", "XGB", "LSTM"]
mae_vals = [mae_rf, mae_xgb, mae_lstm]
rmse_vals = [rmse_rf, rmse_xgb, rmse_lstm]

x = np.arange(len(models))

plt.figure(figsize=(8, 6))
plt.bar(x - w/2, mae_vals, w, label="MAE")
plt.bar(x + w/2, rmse_vals, w, label="RMSE")
plt.xticks(x, models)
plt.legend()
plt.title("All Models Comparison")
plt.show()

# SHAP
X_sample = X_test.iloc[:200]

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_sample, check_additivity=False)

shap.summary_plot(shap_values, X_sample)
